<a href="https://colab.research.google.com/github/abhinandan-084/pytorch-tutorials/blob/main/02_Continous_XOR_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports & Setup

In [1]:
## Standard libraries
import os
import math
import numpy as np
import time

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgba
import seaborn as sns
sns.set()

## Progress bar
from tqdm.notebook import tqdm

/tmp/ipykernel_926/47578708.py:11: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('svg', 'pdf') # For export


In [2]:
# Torch Basics
import torch
print("Using torch", torch.__version__)

# Reproducible Random Numbers
torch.manual_seed(42) # Setting the seed

# Checking GPU Support
gpu_avail = torch.cuda.is_available()
print(f"Is the GPU available? {gpu_avail}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device", device)

# GPU operations have a separate seed we also want to set
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Additionally, some operations on a GPU are implemented stochastic for efficiency
# We want to ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Using torch 2.10.0+cpu
Is the GPU available? False
Device cpu


 In PyTorch, there is a package called **torch.nn** that makes building neural networks more convenient.

 **XOR Model** :  Given two binary inputs and , the label to predict is if either or is while the other is , or the label is in all other cases. The example became famous by the fact that a single neuron, i.e. a linear classifier, cannot learn this simple function. Hence, we will build a small neural network that can learn this function. To make it a little bit more interesting, we move the XOR into continuous space and introduce some gaussian noise on the binary inputs.

In [3]:
import torch.nn as nn
import torch.nn.functional as F

In PyTorch, a neural network is built up out of modules. Modules can contain other modules, and a neural network is considered to be a module itself as well.

The **MyModule class** inherits from **torch.nn.Module** because it is the base class for all neural network modules in PyTorch. Inheriting from nn.Module provides several crucial functionalities and benefits:

* **Parameter Management**: nn.Module automatically tracks all trainable parameters (like weights and biases in nn.Linear layers) that are defined as attributes within the module. This makes it easy to access and update these parameters during training.
* **Submodule Management**: When you define other nn.Module instances (like nn.Linear or nn.Tanh in SimpleClassifier) as attributes, the parent nn.Module automatically registers them as submodules. This allows for hierarchical network structures and proper handling of parameters across the entire network.
Device Transfer: It enables easy movement of the entire model (including all its parameters and submodules) to different devices, such as a GPU (.to(device)).
* **forward Method**: The forward method is where the computation of the module is defined. When you call an instance of an nn.Module (e.g., model(x)), it automatically executes its forward method.
* **Hooks**: nn.Module provides hooks that allow you to customize the behavior of your model during forward and backward passes, which is useful for debugging, visualization, or custom optimizations.

*In essence, nn.Module acts as a blueprint for building neural networks in PyTorch, streamlining the process of creating, managing, and training complex models.*

The basic template of a module is as follows:

In [4]:
class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        # Some init for my module

    def forward(self, x):
        # Function for performing the calculation of the module.
        pass

### Simple Classifier

small_neural_network.svg

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Initialize the modules we need to build the network
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.Tanh()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        return x